In [ ]:
!pip install -q tensorflow scikit-learn seaborn

In [ ]:
import tensorflow as tf
import numpy as np
import time
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
MODEL_PATH = "/content/resnet18_cifar10_float32(2).tflite"

In [ ]:
import tensorflow as tf
import numpy as np

# Load CIFAR-10
(_, _), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# Convert labels from shape (10000, 1) -> (10000,)
y_test = y_test.flatten()

print("Test images:", x_test.shape)
print("Test labels:", y_test.shape)

In [ ]:
# ✏️ NEW: NORMALIZATION, used by BOTH models (TFLite cell and PyTorch cell).
# Must match the transform in the training notebook. A pretrained ResNet-18 is normally
# fine-tuned with the ImageNet values below. If training used something else, change
# them HERE only, then re-run the TFLite and PyTorch cells.
import numpy as np

MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)  # ImageNet
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

# Other common choices:
# MEAN, STD = np.array([0.4914, 0.4822, 0.4465], dtype=np.float32), np.array([0.2470, 0.2435, 0.2616], dtype=np.float32)  # CIFAR-10
# MEAN, STD = np.array([0.5, 0.5, 0.5], dtype=np.float32), np.array([0.5, 0.5, 0.5], dtype=np.float32)                     # [-1, 1]


In [ ]:
import os

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if file.endswith('.tflite'):
            print(os.path.join(root, file))

In [ ]:
import glob

tflite_files = glob.glob(
    "/content/drive/MyDrive/**/*.tflite",
    recursive=True
)

print("Found models:")

for f in tflite_files:
    print(f)


MODEL_PATH = tflite_files[0]

print("Using:")
print(MODEL_PATH)

In [ ]:
interpreter = tf.lite.Interpreter(model_path=MODEL_PATH)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Input details:")
print(input_details)

print("\nOutput details:")
print(output_details)



In [ ]:
print("INPUT DETAILS")
print("Shape:", input_details[0]["shape"])
print("Dtype:", input_details[0]["dtype"])
print("Quantization:", input_details[0]["quantization"])

print("\nOUTPUT DETAILS")
print("Shape:", output_details[0]["shape"])
print("Dtype:", output_details[0]["dtype"])
print("Quantization:", output_details[0]["quantization"])

In [ ]:
import tensorflow as tf
import numpy as np
import time
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt
import seaborn as sns





# ==========================================
# 2. CIFAR-10 CLASS NAMES
# ==========================================

class_names = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck"
]


# ==========================================
# 3. MODEL INFORMATION
# ==========================================

input_info = input_details[0]
output_info = output_details[0]

input_index = input_info["index"]
output_index = output_info["index"]

input_shape = input_info["shape"]
input_dtype = input_info["dtype"]

print("\nModel input shape:", input_shape)
print("Model input dtype:", input_dtype)
print("Model output shape:", output_info["shape"])
print("Model output dtype:", output_info["dtype"])


# ==========================================
# 4. RUN INFERENCE
# ==========================================

predictions = []

start_time = time.time()

for i in range(len(x_test)):

    # Original CIFAR-10 image: 32x32x3
    image = x_test[i]

    # Resize to model input: 224x224
    image = tf.image.resize(
        image,
        (224, 224)
    ).numpy()

    # Add batch dimension
    image = np.expand_dims(image, axis=0)

    # --------------------------------------
    # FLOAT32 MODEL
    # --------------------------------------

    if input_dtype == np.float32:

        image = image.astype(np.float32)

        # Normalize pixel values
        image = image / 255.0

        # ✏️ EDITED: apply the same mean/std normalization used in training
        # (defined once in the "NORMALIZATION" cell). Without it accuracy drops to ~61%.
        image = (image - MEAN) / STD

    else:

        image = image.astype(input_dtype)


    # Set input
    interpreter.set_tensor(
        input_index,
        image
    )

    # Run model
    interpreter.invoke()

    # Get output
    output = interpreter.get_tensor(
        output_index
    )

    # Get predicted class
    predicted_class = np.argmax(output[0])

    predictions.append(predicted_class)


end_time = time.time()


predictions = np.array(predictions)


# ==========================================
# 5. INFERENCE TIME
# ==========================================

total_time = end_time - start_time

avg_time = total_time / len(x_test)

print("\n===================================")
print("INFERENCE RESULTS")
print("===================================")

print(
    f"Total inference time: {total_time:.2f} seconds"
)

print(
    f"Average inference time: {avg_time * 1000:.4f} ms/image"
)

print(
    f"Images processed: {len(x_test)}"
)

In [ ]:
accuracy = accuracy_score(
    y_test,
    predictions
)

print("Accuracy:", accuracy)
print(f"Accuracy: {accuracy * 100:.2f}%")

In [ ]:
precision = precision_score(
    y_test,
    predictions,
    average="weighted",
    zero_division=0
)

recall = recall_score(
    y_test,
    predictions,
    average="weighted",
    zero_division=0
)

f1 = f1_score(
    y_test,
    predictions,
    average="weighted",
    zero_division=0
)

print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

In [ ]:
report = classification_report(
    y_test,
    predictions,
    target_names=class_names,
    digits=4
)

print(report)

In [ ]:
cm = confusion_matrix(
    y_test,
    predictions
)

plt.figure(figsize=(10, 8))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.title("ResNet18 CIFAR-10 Confusion Matrix")

plt.show()

In [ ]:
print("PER-CLASS ACCURACY")
print("=" * 40)

for i, class_name in enumerate(class_names):

    total = np.sum(y_test == i)

    correct = np.sum(
        (y_test == i) &
        (predictions == i)
    )

    class_accuracy = correct / total

    print(
        f"{class_name:12s}: "
        f"{class_accuracy * 100:.2f}%"
    )

In [ ]:
import os

model_size_bytes = os.path.getsize(MODEL_PATH)

model_size_mb = (
    model_size_bytes /
    (1024 * 1024)
)

print(
    f"Model Size: {model_size_mb:.2f} MB"
)

In [ ]:
import os

model_size_bytes = os.path.getsize(MODEL_PATH)

model_size_mb = (
    model_size_bytes /
    (1024 * 1024)
)

print(
    f"Model Size: {model_size_mb:.2f} MB"
)

In [ ]:
results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "Total Inference Time (s)",
        "Average Inference Time (ms)",
        "Model Size (MB)"
    ],

    "Value": [
        accuracy,
        precision,
        recall,
        f1,
        total_time,
        avg_time * 1000,
        model_size_mb
    ]
})

results

In [ ]:
import glob

pth_files = glob.glob(
    "/content/drive/MyDrive/**/*.pth",
    recursive=True
)

for f in pth_files:
    print(f)

In [ ]:
PTH_PATH = [f for f in pth_files if "best_resnet18_cifar10" in f][0]

print(PTH_PATH)

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import resnet18

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = resnet18(weights=None)

# CIFAR-10 = 10 classes
model.fc = nn.Linear(
    model.fc.in_features,
    10
)

checkpoint = torch.load(
    PTH_PATH,
    map_location=device
)

print(type(checkpoint))

In [ ]:
if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
    model.load_state_dict(checkpoint["state_dict"])
else:
    model.load_state_dict(checkpoint)

model = model.to(device)
model.eval()

print("Original ResNet18 loaded successfully!")

In [ ]:
# ✏️ NEW: QUICK CHECK, which normalization did training use?
# Scores the PyTorch model on 500 test images with each option. The right one is the
# clearly highest (expected ~90%+). If it isn't ImageNet, copy it into the
# NORMALIZATION cell and re-run the TFLite and PyTorch cells.
from torchvision import transforms
import torch

candidates = {
    "ImageNet": ([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    "CIFAR-10": ([0.4914, 0.4822, 0.4465], [0.2470, 0.2435, 0.2616]),
    "[-1, 1]": ([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
    "none (/255 only)": None,
}

for name, norm in candidates.items():
    steps = [transforms.ToPILImage(), transforms.Resize((224, 224)), transforms.ToTensor()]
    if norm:
        steps.append(transforms.Normalize(*norm))
    check_transform = transforms.Compose(steps)

    batch = torch.stack([check_transform(img) for img in x_test[:500]]).to(device)
    with torch.no_grad():
        preds = torch.cat([model(batch[i:i + 100]).argmax(1) for i in range(0, 500, 100)]).cpu().numpy()
    print(f"{name:18s}: {(preds == y_test[:500]).mean() * 100:.1f}%")


In [ ]:
from torchvision import transforms
import numpy as np
import time

transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)  # ✏️ EDITED: same normalization as training and the TFLite run
])

print("Total test images:", len(x_test))

In [ ]:
original_predictions = []

start_time = time.time()

with torch.no_grad():

    for image in x_test:

        image = transform(image)

        image = image.unsqueeze(0).to(device)

        output = model(image)

        predicted_class = torch.argmax(
            output,
            dim=1
        ).item()

        original_predictions.append(predicted_class)

end_time = time.time()

original_predictions = np.array(original_predictions)

print("Evaluation completed!")
print("Images processed:", len(original_predictions))
print("Total time:", end_time - start_time, "seconds")

In [ ]:
accuracy = accuracy_score(
    y_test,
    original_predictions
)

precision = precision_score(
    y_test,
    original_predictions,
    average="weighted",
    zero_division=0
)

recall = recall_score(
    y_test,
    original_predictions,
    average="weighted",
    zero_division=0
)

f1 = f1_score(
    y_test,
    original_predictions,
    average="weighted",
    zero_division=0
)

print(f"Accuracy : {accuracy * 100:.2f}%")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

In [ ]:
tflite_predictions = np.array(predictions)

In [ ]:
print("TFLite predictions:", len(predictions))
print("Test labels:", len(y_test))

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Original model
original_accuracy = accuracy_score(
    y_test,
    original_predictions
)

original_precision = precision_score(
    y_test,
    original_predictions,
    average="weighted",
    zero_division=0
)

original_recall = recall_score(
    y_test,
    original_predictions,
    average="weighted",
    zero_division=0
)

original_f1 = f1_score(
    y_test,
    original_predictions,
    average="weighted",
    zero_division=0
)


# TFLite model
tflite_accuracy = accuracy_score(
    y_test,
    tflite_predictions
)

tflite_precision = precision_score(
    y_test,
    tflite_predictions,
    average="weighted",
    zero_division=0
)

tflite_recall = recall_score(
    y_test,
    tflite_predictions,
    average="weighted",
    zero_division=0
)

tflite_f1 = f1_score(
    y_test,
    tflite_predictions,
    average="weighted",
    zero_division=0
)

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ],

    "Original ResNet18": [
        original_accuracy * 100,
        original_precision * 100,
        original_recall * 100,
        original_f1 * 100
    ],

    "TFLite ResNet18": [
        tflite_accuracy * 100,
        tflite_precision * 100,
        tflite_recall * 100,
        tflite_f1 * 100
    ]
})

comparison

In [ ]:
print(len(predictions))